# Retail Banking Customer Churn & Retention Analytics
**Emmanuel Odinamba** | Python · SQL · Statistical Analysis · Machine Learning

This notebook analyzes a public 10,000-customer banking dataset to identify churn patterns and build an interpretable churn-risk model.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import sqlite3

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, roc_auc_score, ConfusionMatrixDisplay, RocCurveDisplay

## 1. Load and inspect the data

In [ ]:
DATA_URL = "https://raw.githubusercontent.com/YBI-Foundation/Dataset/main/Bank%20Churn%20Modelling.csv"
df = pd.read_csv(DATA_URL)

print("Shape:", df.shape)
display(df.head())

## 2. Data-quality review
Before analysis, check missing values, duplicates and data types.

In [ ]:
print("Missing values:", df.isna().sum().sum())
print("Duplicate rows:", df.duplicated().sum())
display(df.dtypes.to_frame("dtype"))

df = df.rename(columns={
    "Num Of Products": "NumOfProducts",
    "Has Credit Card": "HasCreditCard",
    "Is Active Member": "IsActiveMember",
    "Estimated Salary": "EstimatedSalary"
})
df = df.drop(columns=["CustomerId", "Surname"], errors="ignore")

## 3. Exploratory analysis
Measure the portfolio churn KPI and identify segments with elevated churn.

In [ ]:
print(f"Overall churn rate: {df['Churn'].mean():.2%}")

for col in ["Geography", "Gender", "IsActiveMember", "NumOfProducts"]:
    out = df.groupby(col)["Churn"].agg(customers="size", churn_rate="mean")
    out["churn_rate"] = (out["churn_rate"] * 100).round(2)
    display(out.sort_values("churn_rate", ascending=False))

In [ ]:
df["AgeBand"] = pd.cut(
    df["Age"],
    bins=[17, 29, 45, 60, 100],
    labels=["Under 30", "30-45", "46-60", "60+"]
)

age = df.groupby("AgeBand", observed=False)["Churn"].agg(customers="size", churn_rate="mean")
age["churn_rate"] = (age["churn_rate"] * 100).round(2)
display(age)

## 4. SQL analysis
Load the analytical table into SQLite and answer a business question using SQL.

In [ ]:
conn = sqlite3.connect(":memory:")
df.drop(columns=["AgeBand"]).to_sql("customers", conn, index=False, if_exists="replace")

query = """
SELECT
    Geography,
    COUNT(*) AS customers,
    ROUND(100.0 * AVG(Churn), 2) AS churn_rate_pct,
    ROUND(AVG(Balance), 2) AS avg_balance
FROM customers
GROUP BY Geography
ORDER BY churn_rate_pct DESC;
"""

display(pd.read_sql_query(query, conn))

## 5. Interpretable churn model
Use a stratified 80/20 train-test split and logistic regression. Class weighting helps account for the smaller churn class.

In [ ]:
X = df.drop(columns=["Churn", "AgeBand"])
y = df["Churn"]

categorical = ["Geography", "Gender"]
numeric = [c for c in X.columns if c not in categorical]

preprocessor = ColumnTransformer([
    ("numeric", StandardScaler(), numeric),
    ("categorical", OneHotEncoder(handle_unknown="ignore"), categorical)
])

model = Pipeline([
    ("preprocess", preprocessor),
    ("model", LogisticRegression(max_iter=1000, class_weight="balanced", random_state=42))
])

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

model.fit(X_train, y_train)
pred = model.predict(X_test)
prob = model.predict_proba(X_test)[:, 1]

print(classification_report(y_test, pred, digits=3))
print("ROC-AUC:", round(roc_auc_score(y_test, prob), 3))

## 6. Model evaluation
A confusion matrix shows correct and incorrect classifications. The ROC curve summarizes how well the model separates churners from non-churners across thresholds.

In [ ]:
ConfusionMatrixDisplay.from_predictions(y_test, pred)
plt.title("Churn Model Confusion Matrix")
plt.show()

RocCurveDisplay.from_predictions(y_test, prob)
plt.title("ROC Curve — Customer Churn Model")
plt.show()

## 7. Interpretation and business use
The model is a prioritization tool, not proof of causality. Segment-level findings should trigger deeper investigation. A practical retention workflow would combine churn probability with customer value, test targeted interventions, and measure incremental retention.